# Premium Model Training
Predicts insurance premium prices based on gig worker risk factors

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import os

In [ ]:
# Generate synthetic training data
np.random.seed(42)
n_samples = 10000

data = {
    'coverage_amount': np.random.uniform(1000, 50000, n_samples),
    'occupation_risk': np.random.uniform(0.3, 1.0, n_samples),
    'zone_risk': np.random.uniform(0.2, 1.0, n_samples),
    'hours_worked': np.random.uniform(20, 80, n_samples),
    'monthly_earnings': np.random.uniform(1000, 8000, n_samples)
}

df = pd.DataFrame(data)

# Calculate premium (ground truth)
base_rate = 0.07
df['premium'] = (
    df['coverage_amount'] * base_rate * 
    df['occupation_risk'] * df['zone_risk'] *
    (1 + 0.001 * df['hours_worked']) *
    (1 - 0.00005 * df['monthly_earnings'])
)

df.head()

In [ ]:
# Prepare features and target
feature_cols = ['coverage_amount', 'occupation_risk', 'zone_risk', 'hours_worked', 'monthly_earnings']
X = df[feature_cols]
y = df['premium']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Train Gradient Boosting model
model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)
print("Model trained successfully!")

In [ ]:
# Evaluate model
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: ${mae:.2f}")
print(f"Root Mean Squared Error: ${rmse:.2f}")
print(f"R² Score: {r2:.4f}")

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance:")
print(importance.to_string(index=False))

In [ ]:
# Save model
os.makedirs('models', exist_ok=True)
model_path = 'models/premium_model.joblib'
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

In [ ]:
# Test prediction
test_features = {
    'coverage_amount': 10000,
    'occupation_risk': 0.7,
    'zone_risk': 0.5,
    'hours_worked': 40,
    'monthly_earnings': 3000
}

feature_array = np.array([list(test_features.values())])
prediction = model.predict(feature_array)[0]

print(f"Test Prediction: ${prediction:.2f}")